# Notebook 2: Hiperspektral Veri Setlerinin Ön İşlenmesi ve Keşfi

Bu notebook'ta **Notebook 1**'de kaydedilen veri setleri yüklenir ve şu işlemler yapılır:
1. **Veri Keşfi:** Temel istatistikler, boyutlar, sınıf dağılımları
2. **Gürültü Azaltma:** Gaussian filtreleme
3. **Boyut İndirgeme:** PCA (Temel Bileşen Analizi)
4. **Normalizasyon:** StandardScaler ile özellik normalleştirmesi
5. **Train/Test Split:** Veri setini eğitim ve test setlerine ayırma
6. **Görselleştirme:** Spektral bantlar, sınıf haritaları, dağılımlar
7. **Veri Kaydetme:** İşlenmiş verileri kaydetme

⚠️ **ÖNKOŞULu:** Lütfen önce `01_dataset_loading.ipynb` notebook'unu çalıştırın.

## 1. Gerekli Kütüphanelerin İmport Edilmesi

In [ ]:
# Temel kütüphaneler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import ndimage, io
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings('ignore')

# Makine öğrenmesi kütüphaneleri
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# İşletim sistemi
,
,
,
,
,
,
8
,
10
,
,
✓ Tüm kütüphaneler başarıyla yüklendi!")

## 2. Google Drive Montajı ve Veri Klasörleri

In [ ]:
# Google Drive'ı monte et
from google.colab import drive
drive.mount('/content/drive')

# Veri klasörleri
BASE_DIR = '/content/drive/MyDrive/hyperspectral_datasets/'
RAW_DIR = os.path.join(BASE_DIR, 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed')
PREPROCESSED_DIR = os.path.join(BASE_DIR, 'preprocessed')

os.makedirs(PREPROCESSED_DIR, exist_ok=True)

print(f"✓ Google Drive bağlandı.")
print(f"Veri Klasörleri:\"")
print(f"  Processed Data: {PROCESSED_DIR}\")\n")
print(f"  Preprocessed Data: {PREPROCESSED_DIR}")

## 3. Veri Setlerini Yükleme

In [ ]:
# Veri setlerini yükle (Notebook 1'den kaydedilmiş)
datasets = {}

# Indian Pines
try:
    indian_img = np.load(os.path.join(PROCESSED_DIR, 'indian_pines_img.npy'))
    indian_gt = np.load(os.path.join(PROCESSED_DIR, 'indian_pines_gt.npy'))
    datasets['Indian Pines'] = {
        'img': indian_img,
        'gt': indian_gt,
        'shape': indian_img.shape
    }
    print(f"✓ Indian Pines yüklendi: {indian_img.shape}")
except FileNotFoundError:
    print("✗ Indian Pines bulunamadı. Lütfen 01_dataset_loading.ipynb'ı çalıştırın.")

# Pavia University
try:
    pavia_img = np.load(os.path.join(PROCESSED_DIR, 'pavia_university_img.npy'))
    pavia_gt = np.load(os.path.join(PROCESSED_DIR, 'pavia_university_gt.npy'))
    datasets['Pavia University'] = {
        'img': pavia_img,
        'gt': pavia_gt,
        'shape': pavia_img.shape
    }
    print(f"✓ Pavia University yüklendi: {pavia_img.shape}")
except FileNotFoundError:
    print("✗ Pavia University bulunamadı. Lütfen 01_dataset_loading.ipynb'ı çalıştırın.")

# Salinas
try:
    salinas_img = np.load(os.path.join(PROCESSED_DIR, 'salinas_img.npy'))
    salinas_gt = np.load(os.path.join(PROCESSED_DIR, 'salinas_gt.npy'))
    datasets['Salinas'] = {
        'img': salinas_img,
        'gt': salinas_gt,
        'shape': salinas_img.shape
    }
    print(f"✓ Salinas yüklendi: {salinas_img.shape}")
except FileNotFoundError:
    print("✗ Salinas bulunamadı. Lütfen 01_dataset_loading.ipynb'ı çalıştırın.")

print(f"\n✓ Toplam {len(datasets)} veri seti yüklendi.")

## 4. Veri Keşfi ve Temel İstatistikler

In [ ]:
# Her veri seti için temel istatistikler
for dataset_name, dataset_info in datasets.items():
    img = dataset_info['img']
    gt = dataset_info['gt']
    
    print(f"\n{'='*60}")
    print(f"{dataset_name}")
    print(f"{'='*60}")
    print(f"Görüntü Şekli: {img.shape}")
    print(f"Piksel Boyutu: {img.shape[0]} × {img.shape[1]}")
    print(f"Spektral Bant Sayısı: {img.shape[2]}")
    print(f"\nVeri Türü: {img.dtype}")
    print(f"Min Değer: {img.min():.4f}")
    print(f"Max Değer: {img.max():.4f}")
    print(f"Ortalama: {img.mean():.4f}")
    print(f"Standart Sapma: {img.std():.4f}")
    
    print(f"\nSınıf Bilgileri:")
    unique_classes = np.unique(gt)
    unique_classes = unique_classes[unique_classes > 0]
    print(f"Sınıf Sayısı: {len(unique_classes)}")
    print(f"Etiketli Piksel Sayısı: {np.sum(gt > 0)}")
    print(f"Etiketli Piksel Oranı: {np.sum(gt > 0) / (gt.shape[0] * gt.shape[1]) * 100:.2f}%")

## 5. Gürültü Azaltma (Gaussian Filtreleme)

In [ ]:
def apply_gaussian_filter(img, sigma=1.0):
    """
    Hiperspektral görüntüye Gaussian filtresi uygula
    
    Args:
        img: Hiperspektral görüntü (H × W × B)
        sigma: Gaussian çekirdeğinin standart sapması
    
    Returns:
        Filtrelenmiş görüntü
    """
    filtered_img = np.zeros_like(img)
    
    for band in range(img.shape[2]):
        filtered_img[:, :, band] = gaussian_filter(img[:, :, band], sigma=sigma)
    
    return filtered_img


# Tüm veri setlerine Gaussian filtresi uygula
filtered_datasets = {}

print("Gaussian filtresi uygulanıyor...")
for dataset_name, dataset_info in datasets.items():
    print(f"  {dataset_name}...", end=" ")
    filtered_img = apply_gaussian_filter(dataset_info['img'], sigma=1.0)
    filtered_datasets[dataset_name] = {
        'img': filtered_img,
        'gt': dataset_info['gt'],
        'shape': filtered_img.shape
    }
    print("✓")

print("\n✓ Gaussian filtresi uygulandı.")

## 6. Veriyi Düzleştirme ve Şekil Değiştirme

In [ ]:
# Veriyi sınıflandırma için uygun şekle getir
reshaped_datasets = {}

for dataset_name, dataset_info in filtered_datasets.items():
    img = dataset_info['img']
    gt = dataset_info['gt']
    
    # (H, W, B) -> (H*W, B)
    img_reshaped = img.reshape(-1, img.shape[2])
    gt_reshaped = gt.reshape(-1)
    
    reshaped_datasets[dataset_name] = {
        'img': img_reshaped,
        'gt': gt_reshaped,
        'img_3d': img,
        'shape_2d': img_reshaped.shape,
        'shape_3d': img.shape
    }
    
    print(f"{dataset_name}: {img.shape} -> {img_reshaped.shape}")

## 7. PCA ile Boyut İndirgeme